# Lab | Chains in LangChain

## Outline

* LLMChain
* Sequential Chains
  * SimpleSequentialChain
  * SequentialChain
* Router Chain

In [1]:
import warnings
warnings.filterwarnings('ignore')

In [25]:
import os
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate, PromptTemplate
from langchain.chains import (
    LLMChain,
    SimpleSequentialChain,
    SequentialChain,
)
from langchain.chains.router import MultiPromptChain
from langchain.chains.router.llm_router import (
    LLMRouterChain,
    RouterOutputParser,
)

# Define the path BEFORE using it.
env_path = Path.cwd() / ".env"

print(".env location:", env_path)
print(".env found:", env_path.is_file())

load_dotenv(dotenv_path=env_path)

if not os.getenv("OPENAI_API_KEY", "").strip():
    raise ValueError(f"OPENAI_API_KEY was not loaded from {env_path}")

llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0.5,
)

print("API key loaded and model initialized.")

.env location: /home/haripriya/ironhack/15_ai_engineering/lab-chains-in-langchain/.env
.env found: True
API key loaded and model initialized.


In [3]:
#!pip install pandas

In [5]:
import pandas as pd
df = pd.read_csv('./data/Data.csv')

In [6]:
df.head()

,Product,Review
0,Queen Size Sheet Set,I ordered a king size set. My only criticism w...
1,Waterproof Phone Pouch,"I loved the waterproof sac, although the openi..."
2,Luxury Air Mattress,This mattress had a small hole in the top of i...
3,Pillows Insert,This is the best throw pillow fillers on Amazo...
4,Milk Frother Handheld,I loved this product. But they only seem to la...


## LLMChain

In [6]:
!pip install langchain_community

  Using cached truststore-0.10.5-py3-none-any.whl.metadata (4.5 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 9.1 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 9.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 574.7/574.7 kB 10.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.6/883.6 kB 13.0 MB/s  0:00:00
Using cached truststore-0.10.5-py3-none-any.whl (19 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 16.8 MB/s  0:00:00 eta 0:00:01
  Attempting uninstall: langsmith╺━━━━━━━━━━━━━━━  9/15 [httpx2]c-settings]
    Found existing installation: langsmith 0.1.147━━━━━━━━━━━━  9/15 [httpx2]
    Uninstalling langsmith-0.1.147:0m╺━━━━━━━━━━━━━━━  9/15 [httpx2]
      Successfully uninstalled langsmith-0.1.147━━━━━━━━━━━━━━  9/15 [httpx2]
  Attempting uninstall: langchain-core╸━━━━━━━━━━━━━ 10/15 [langsmith]
    Found existing installation: langchain-core 0.2.43━━━━━━━━ 10/15 [langsmith]
    Uninstalling langch

In [27]:
from langchain_openai import ChatOpenAI
from langchain.prompts import ChatPromptTemplate
from langchain.chains import LLMChain

In [26]:
#Replace None by your own value and justify
#llm = ChatOpenAI(temperature=None)


In [28]:
prompt = ChatPromptTemplate.from_template(     "Write a short description of {product}. "
    "Explain its purpose, typical uses, and main benefits. "
    "Do not invent brand-specific specifications."
)

In [29]:

chain = LLMChain(llm=llm, prompt=prompt)

In [30]:
print("Model used:", chain.llm.model_name)

result = chain.invoke({
    "product": "a reusable water bottle",
})

print(result["text"])

Model used: gpt-4o-mini
A reusable water bottle is a portable container designed to hold liquids, primarily water, for hydration on the go. Typically made from materials such as stainless steel, glass, or BPA-free plastic, these bottles come in various sizes and designs to suit different lifestyles and preferences. 

The primary purpose of a reusable water bottle is to provide a sustainable alternative to single-use plastic bottles, helping to reduce environmental waste. They are commonly used by individuals during outdoor activities like hiking, biking, or exercising, as well as in everyday settings such as the office, gym, or school.

The main benefits of using a reusable water bottle include promoting hydration, saving money by reducing the need to purchase bottled water, and minimizing plastic waste. Additionally, many reusable bottles are designed to keep beverages cold or hot for extended periods, offering convenience and enhancing the drinking experience. By choosing a reusable 

## SimpleSequentialChain

In [31]:
from langchain.chains import SimpleSequentialChain

In [38]:
llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0.9,
)

# prompt template 1
first_prompt = ChatPromptTemplate.from_template(
      "Write a short description of {product}. "
    "Focus on typical uses and avoid unsupported specifications."
)

# Chain 1
chain_one = LLMChain(llm=llm, prompt=first_prompt)

In [39]:

# prompt template 2
second_prompt = ChatPromptTemplate.from_template(
    "Using this product description, write a two-sentence advertisement. "
    "Do not add new factual claims.\n\n"
    "Description: {description}"
)
# chain 2
chain_two = LLMChain(llm=llm, prompt=second_prompt)

In [40]:
overall_simple_chain = SimpleSequentialChain(chains=[chain_one, chain_two],
                                             verbose=True
                                            )

In [41]:
result = overall_simple_chain.invoke({
    "input": "a reusable water bottle",
})

print(result["output"])



> Entering new SimpleSequentialChain chain...
This reusable water bottle is designed for everyday convenience and sustainability. Ideal for staying hydrated during workouts, commutes, or outdoor adventures, it features a sturdy construction that can handle a variety of activities. Its leak-proof lid ensures that you can toss it in your bag without worry, while the wide mouth opening makes it easy to fill, clean, and add ice. Perfect for both cold water and hot beverages, it’s a versatile companion for the gym, hiking, or simply keeping on your desk at work. With its eco-friendly design, it helps reduce single-use plastic waste, making it a responsible choice for the environment.
Stay hydrated on the go with our versatile, eco-friendly reusable water bottle, perfect for workouts, commutes, and outdoor adventures. With a leak-proof lid and wide mouth opening, it's designed for your convenience while helping to reduce single-use plastic waste.

> Finished chain.
Stay hydrated on the go 

**Repeat the above twice for different products**

In [42]:
for product in ["a yoga mat", "a desk organizer"]:
    result = overall_simple_chain.invoke({"input": product})

    print("\nProduct:", product)
    print("Advertisement:", result["output"])



> Entering new SimpleSequentialChain chain...
A yoga mat is a versatile and essential accessory for anyone practicing yoga, providing a comfortable and supportive surface for a variety of poses. It helps to enhance stability and grip, allowing practitioners to focus on their alignment and breathing without the distraction of slipping or discomfort. Ideal for different styles of yoga—whether it's a gentle restorative session or a more intense power flow—yoga mats often feature cushioning to protect joints while ensuring enough firmness for balance. They are typically lightweight and portable, making them easy to transport to classes or outdoor sessions. Beyond yoga, these mats can also be used for other exercises, stretching routines, or meditation practices, making them a popular choice for anyone interested in physical fitness and mindfulness.
Elevate your practice with our versatile yoga mat, designed to provide comfort and support for all yoga styles. Lightweight and portable, it’

## SequentialChain

In [43]:
from langchain.chains import SequentialChain

In [44]:
llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0.9,
)

first_prompt = ChatPromptTemplate.from_template(
    "Translate the following review into English. "
    "If it is already English, return it unchanged. "
    "Return only the review text.\n\n"
    "Review: {Review}"
)

chain_one = LLMChain(llm=llm, prompt=first_prompt, 
                     output_key="English_Review",
                    )


In [45]:
second_prompt = ChatPromptTemplate.from_template(
      "Summarize this review in one English sentence. "
    "Preserve its main praise or complaint.\n\n"
    "Review: {English_Review}"
)

chain_two = LLMChain(llm=llm, prompt=second_prompt, 
                     output_key="summary",
                    )


In [46]:
# prompt template 3: translate to english or other language
third_prompt = ChatPromptTemplate.from_template(
        "Identify the language of the following review. "
    "Return only the language name in English.\n\n"
    "Review: {Review}"
)
# chain 3: input= Review and output= language
chain_three = LLMChain(llm=llm, prompt=third_prompt,
                       output_key="language",
                      )


In [47]:

# prompt template 4: follow up message that take as inputs the two previous prompts' variables
fourth_prompt = ChatPromptTemplate.from_template(
          "Write a short, polite customer-service reply in {language}, "
    "based on this review summary: {summary}\n"
    "Acknowledge the feedback. "
    "Do not invent refunds, replacements, or company policies."
)
chain_four = LLMChain(llm=llm, prompt=fourth_prompt,
                      output_key="followup_message",
                     )


In [48]:
# overall_chain: input= Review 
# and output= English_Review,summary, followup_message
overall_chain = SequentialChain(
    chains=[chain_one, chain_two, chain_three, chain_four],
    input_variables=["Review"],
    output_variables=[  "English_Review",
        "summary",
        "followup_message",],
    verbose=True
)

In [49]:
review = (
    "Die Flasche sieht schön aus, "
    "aber der Deckel ist nach zwei Tagen kaputt gegangen."
)

result = overall_chain.invoke({"Review": review})

print("English review:", result["English_Review"])
print("Summary:", result["summary"])
print("Reply:", result["followup_message"])



> Entering new SequentialChain chain...

> Finished chain.
English review: The bottle looks nice, but the cap broke after two days.
Summary: The bottle is visually appealing, but the cap broke within two days.
Reply: Sehr geehrte/r Kunde/in,

vielen Dank für Ihr Feedback zu unserer Flasche. Es freut uns, dass Ihnen das Design gefällt. Es tut uns leid zu hören, dass der Deckel innerhalb von zwei Tagen beschädigt ist. Wir werden Ihr Anliegen intern prüfen, um die Qualität unserer Produkte weiter zu verbessern.

Mit freundlichen Grüßen,  
Ihr Kundenservice-Team


**Repeat the above twice for different products or reviews**

In [50]:
for review in df["Review"].dropna().iloc[:2]:
    result = overall_chain.invoke({"Review": review})

    print("\nOriginal:", review)
    print("Summary:", result["summary"])
    print("Reply:", result["followup_message"])



> Entering new SequentialChain chain...

> Finished chain.

Original: I ordered a king size set. My only criticism would be that I wish seller would offer the king size set with 4 pillowcases. I separately ordered a two pack of pillowcases so I could have a total of four. When I saw the two packages, it looked like the color did not exactly match. Customer service was excellent about sending me two more pillowcases so I would have four that matched. Excellent! For the cost of these sheets, I am satisfied with the characteristics and coolness of the sheets.
Summary: The reviewer is satisfied with the quality and coolness of the king size sheets but wishes the seller offered a set with four matching pillowcases, although they appreciated the excellent customer service in resolving the issue.
Reply: Dear [Customer's Name],

Thank you for your thoughtful feedback! We're delighted to hear that you are satisfied with the quality of our king size sheets and appreciated our customer service.

## Router Chain

In [51]:
physics_template = """You are a very smart physics professor. \
You are great at answering questions about physics in a concise\
and easy to understand manner. \
When you don't know the answer to a question you admit\
that you don't know.

Here is a question:
{input}"""


math_template = """You are a very good mathematician. \
You are great at answering math questions. \
You are so good because you are able to break down \
hard problems into their component parts, 
answer the component parts, and then put them together\
to answer the broader question.

Here is a question:
{input}"""

history_template = """You are a very good historian. \
You have an excellent knowledge of and understanding of people,\
events and contexts from a range of historical periods. \
You have the ability to think, reflect, debate, discuss and \
evaluate the past. You have a respect for historical evidence\
and the ability to make use of it to support your explanations \
and judgements.

Here is a question:
{input}"""


computerscience_template = """ You are a successful computer scientist.\
You have a passion for creativity, collaboration,\
forward-thinking, confidence, strong problem-solving capabilities,\
understanding of theories and algorithms, and excellent communication \
skills. You are great at answering coding questions. \
You are so good because you know how to solve a problem by \
describing the solution in imperative steps \
that a machine can easily interpret and you know how to \
choose a solution that has a good balance between \
time complexity and space complexity. 

Here is a question:
{input}"""

biology_template = """You are an excellent biologist. \
You have a deep understanding of living organisms, \
from the molecular and cellular level to entire ecosystems. \
You are skilled at observing patterns in nature, analyzing biological data, \
and explaining complex processes like evolution, genetics, physiology, and ecology. \
You can clearly communicate how life functions and adapts, \
and you make connections between different biological concepts \
to answer challenging questions.

Here is a question:
{input}"""

In [52]:
prompt_infos = [
    {
        "name": "physics", 
        "description": "Good for answering questions about physics", 
        "prompt_template": physics_template
    },
    {
        "name": "math", 
        "description": "Good for answering math questions", 
        "prompt_template": math_template
    },
    {
        "name": "History", 
        "description": "Good for answering history questions", 
        "prompt_template": history_template
    },
    {
        "name": "computer science", 
        "description": "Good for answering computer science questions", 
        "prompt_template": computerscience_template
    },
    {
        "name": "biology",
        "description": "Good for answering biology questions",
        "prompt_template": biology_template
    }
]

In [62]:
from langchain.chains.router import MultiPromptChain
from langchain.chains.router.llm_router import LLMRouterChain,RouterOutputParser
from langchain.prompts import PromptTemplate

In [70]:
llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0,
)

In [64]:
destination_chains = {}
for p_info in prompt_infos:
    name = p_info["name"]
    prompt_template = p_info["prompt_template"]
    prompt = ChatPromptTemplate.from_template(template=prompt_template)
    chain = LLMChain(llm=llm, prompt=prompt)
    destination_chains[name] = chain  
    
destinations = [f"{p['name']}: {p['description']}" for p in prompt_infos]
destinations_str = "\n".join(destinations)

In [65]:
default_prompt = ChatPromptTemplate.from_template("{input}")
default_chain = LLMChain(llm=llm, prompt=default_prompt)

In [66]:
MULTI_PROMPT_ROUTER_TEMPLATE = """Given a raw text input to a \
language model select the model prompt best suited for the input. \
You will be given the names of the available prompts and a \
description of what the prompt is best suited for. \
You may also revise the original input if you think that revising\
it will ultimately lead to a better response from the language model.

<< FORMATTING >>
Return a markdown code snippet with a JSON object formatted to look like:
```json
{{{{
    "destination": string \ name of the prompt to use or "DEFAULT"
    "next_inputs": string \ a potentially modified version of the original input
}}}}
```

REMEMBER: "destination" MUST be one of the candidate prompt \
names specified below OR it can be "DEFAULT" if the input is not\
well suited for any of the candidate prompts.
REMEMBER: "next_inputs" can just be the original input \
if you don't think any modifications are needed.

<< CANDIDATE PROMPTS >>
{destinations}

<< INPUT >>
{{input}}

<< OUTPUT (remember to include the ```json)>>"""

In [71]:
router_template = MULTI_PROMPT_ROUTER_TEMPLATE.format(
    destinations=destinations_str
)
router_prompt = PromptTemplate(
    template=router_template,
    input_variables=["input"],
    output_parser=RouterOutputParser(),
)

router_chain = LLMRouterChain.from_llm(llm, router_prompt)

In [72]:
chain = MultiPromptChain(router_chain=router_chain, 
                         destination_chains=destination_chains, 
                         default_chain=default_chain, verbose=True
                        )

In [73]:
chain.run("What is black body radiation?")



> Entering new MultiPromptChain chain...
physics: {'input': 'What is black body radiation?'}
> Finished chain.


"Black body radiation refers to the electromagnetic radiation emitted by an idealized perfect absorber and emitter of radiation, known as a black body. This concept is essential in understanding thermal radiation.\n\nA black body absorbs all incident radiation, regardless of frequency or angle, and re-emits energy as thermal radiation based on its temperature. The key characteristics of black body radiation are:\n\n1. **Spectral Distribution**: The intensity of the emitted radiation varies with wavelength and temperature. According to Planck's law, this distribution has a specific shape that depends solely on the temperature of the black body. As temperature increases, the peak wavelength of emitted radiation shifts to shorter wavelengths (Wien's displacement law).\n\n2. **Total Emission**: The total energy emitted per unit surface area of the black body is given by the Stefan-Boltzmann law, which states that this energy is proportional to the fourth power of the black body's absolute 

In [74]:
chain.run("what is 2 + 2")



> Entering new MultiPromptChain chain...
math: {'input': 'what is 2 + 2'}
> Finished chain.


"To break down the problem \\(2 + 2\\), we can think about it in component parts:\n\n1. We have the number 2.\n2. We are adding another 2 to it.\n\nNow, let's perform the addition:\n\n- Start with 2.\n- Add another 2: \\(2 + 2 = 4\\).\n\nPutting it all together, the answer to \\(2 + 2\\) is \\(4\\)."

In [75]:
chain.run("Why does every cell in our body contain DNA?")



> Entering new MultiPromptChain chain...
biology: {'input': 'Why does every cell in our body contain DNA?'}
> Finished chain.


"Every cell in our body contains DNA because DNA serves as the blueprint for the development, function, and reproduction of all living organisms. Here are several reasons explaining why this is the case:\n\n1. **Genetic Information**: DNA (deoxyribonucleic acid) contains the genetic instructions necessary for the growth, development, functioning, and reproduction of organisms. It encodes the information that determines the traits and characteristics of an organism, from physical features to metabolic processes.\n\n2. **Cellular Function**: Each cell carries a full set of genetic material, allowing it to produce the proteins and enzymes necessary for its specific functions. While different cell types (e.g., muscle cells, neurons, skin cells) perform different roles, they all rely on the instructions within their DNA to carry out those functions effectively.\n\n3. **Gene Expression**: Although all cells contain the same DNA, not all genes are expressed in every cell. The process of gene 

**Repeat the above at least once for different inputs and chains executions - Be creative!**

In [76]:
questions = [
    "Why do astronauts float inside the International Space Station?",
    "A jacket costs 80 euros and has a 25% discount. What is the final price?",
    "How did the invention of the printing press change Europe?",
    "Explain Python dictionaries with a simple example.",
    "Why do some trees lose their leaves in autumn?",
    "Write a friendly birthday greeting for my friend.",
]

for question in questions:
    print("\nQUESTION:", question)

    result = chain.invoke({"input": question})

    print("ANSWER:", result["text"])
    print("-" * 60)


QUESTION: Why do astronauts float inside the International Space Station?


> Entering new MultiPromptChain chain...
physics: {'input': 'Why do astronauts float inside the International Space Station?'}
> Finished chain.
ANSWER: Astronauts float inside the International Space Station (ISS) because they are in a state of free fall, often referred to as "microgravity." The ISS orbits Earth at a high speed, about 28,000 kilometers per hour (17,500 miles per hour). As the station moves forward, it is also being pulled towards Earth by gravity. However, because it's moving so fast, it keeps missing the Earth, creating a continuous free-fall condition.

In this situation, both the ISS and everything inside it, including the astronauts, are falling towards Earth at the same rate. This means that the astronauts feel weightless, as they are not exerting any force on the ISS or the station itself—similar to what you would experience if you jumped from a high place and fell while freely accelera